# SprintWell — Benchmark analysis

Reads the raw benchmark metrics (`benchmarks/results/raw.csv`, produced by
`benchmarks/scripts/run_benchmark.py`) and renders the figures for chapter 7 of
the thesis (brief §13.2):

1. **Solve time** per algorithm × scale.
2. **Objective value** per equity mode.
3. **Happiness** mean / min / max per algorithm.
4. **Soft-rule satisfaction (%)** per algorithm × scale.

Every figure is exported to both **PNG** and **PDF** under
`benchmarks/results/figures/`. Rows where a solver returned no feasible solution
(`TIMEOUT` with empty metrics) parse as `NaN` and are simply skipped by the
means — the bars stay honest.

> Re-run top-to-bottom after a fresh `raw.csv`. The same CSV always yields the
> same figures.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# When executed (nbconvert) the working directory is this notebook's folder.
RESULTS = (Path.cwd() / ".." / "results").resolve()
RAW_CSV = RESULTS / "raw.csv"
FIGURES = RESULTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

if not RAW_CSV.exists():
    raise FileNotFoundError(
        f"{RAW_CSV} not found — run benchmarks/scripts/run_benchmark.py first."
    )

SCALE_ORDER = ["s1_small", "s2_medium", "s3_large", "s4_xl"]
ALGO_ORDER = ["cpsat", "random", "greedy"]
MODE_ORDER = ["utilitarian", "max-min", "nash"]

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})


def save(fig: plt.Figure, name: str) -> None:
    """Export a figure to PNG and PDF under results/figures/."""
    for ext in ("png", "pdf"):
        fig.savefig(FIGURES / f"{name}.{ext}", bbox_inches="tight")
    print(f"saved figures/{name}.png and figures/{name}.pdf")


df = pd.read_csv(RAW_CSV)
df["scale"] = pd.Categorical(df["scale"], categories=SCALE_ORDER, ordered=True)
print(f"{len(df)} rows | scales: {sorted(df['scale'].dropna().unique())}")
df["status"].value_counts()

## Figure 1 — Solve time by algorithm × scale

Mean solver wall-clock (ms, log scale). This is the headline scalability story:
CP-SAT pays for optimality, the heuristics are near-instant.

In [ ]:
pivot = (
    df.pivot_table(index="scale", columns="algorithm", values="wall_time_ms", aggfunc="mean")
    .reindex(index=SCALE_ORDER)
    .reindex(columns=ALGO_ORDER)
)
fig, ax = plt.subplots(figsize=(8, 5))
pivot.plot(kind="bar", ax=ax, logy=True)
ax.set_xlabel("Scale")
ax.set_ylabel("Mean wall time (ms, log)")
ax.set_title("Solve time by algorithm and scale")
ax.legend(title="Algorithm")
plt.xticks(rotation=0)
save(fig, "time_by_algorithm_scale")
plt.show()

## Figure 2 — Objective value by equity mode

Mean objective value per equity aggregation, grouped by algorithm. Shows how the
utilitarian / max-min / Nash objectives reshape the optimum.

In [ ]:
pivot = (
    df.pivot_table(index="equity_mode", columns="algorithm", values="objective_value", aggfunc="mean")
    .reindex(index=MODE_ORDER)
    .reindex(columns=ALGO_ORDER)
)
fig, ax = plt.subplots(figsize=(8, 5))
pivot.plot(kind="bar", ax=ax)
ax.set_xlabel("Equity mode")
ax.set_ylabel("Mean objective value")
ax.set_title("Objective value by equity mode")
ax.legend(title="Algorithm")
plt.xticks(rotation=0)
save(fig, "objective_by_equity_mode")
plt.show()

## Figure 3 — Happiness (mean / min / max) by algorithm

Average of the per-instance happiness statistics. The gap between *min* and
*mean* is the equity story — CP-SAT under max-min/Nash should lift the floor.

In [ ]:
agg = (
    df.groupby("algorithm")[["happiness_mean", "happiness_min", "happiness_max"]]
    .mean()
    .reindex(ALGO_ORDER)
)
fig, ax = plt.subplots(figsize=(8, 5))
agg.plot(kind="bar", ax=ax)
ax.set_xlabel("Algorithm")
ax.set_ylabel("Happiness (0–1)")
ax.set_title("Happiness mean / min / max by algorithm")
ax.legend(["mean", "min", "max"], title="Statistic")
plt.xticks(rotation=0)
save(fig, "happiness_by_algorithm")
plt.show()

## Figure 4 — Soft-rule satisfaction by algorithm × scale

Mean percentage of soft rules satisfied. This is the quality axis where CP-SAT
should dominate the baselines.

In [ ]:
pivot = (
    df.pivot_table(index="scale", columns="algorithm", values="rules_satisfied_pct", aggfunc="mean")
    .reindex(index=SCALE_ORDER)
    .reindex(columns=ALGO_ORDER)
)
fig, ax = plt.subplots(figsize=(8, 5))
pivot.plot(kind="bar", ax=ax)
ax.set_xlabel("Scale")
ax.set_ylabel("Soft rules satisfied (%)")
ax.set_title("Soft-rule satisfaction by algorithm and scale")
ax.set_ylim(0, 100)
ax.legend(title="Algorithm")
plt.xticks(rotation=0)
save(fig, "rules_satisfied_by_algorithm_scale")
plt.show()

## Summary table

Per scale × algorithm means for the key metrics, also written to
`results/analysis_summary.csv` for the thesis tables.

In [ ]:
summary = (
    df.groupby(["scale", "algorithm"], observed=True)
    .agg(
        wall_time_ms=("wall_time_ms", "mean"),
        objective_value=("objective_value", "mean"),
        happiness_mean=("happiness_mean", "mean"),
        rules_satisfied_pct=("rules_satisfied_pct", "mean"),
        deadlines_met_pct=("deadlines_met_pct", "mean"),
    )
    .round(2)
)
summary.to_csv(RESULTS / "analysis_summary.csv")
summary